# Sparkle V2 — Fase 3.6f: CIERRE. Evaluación única sobre el Test oficial

**Protocolo:** `Protocolo_Fase36_Camino_al_70.md` **Enmienda 5** (2026-07-29).

> ⚠️ **Este notebook consume la única evaluación sobre el Test oficial de Fase 3.6.**
> Todo está congelado por la Enmienda 5. Nada puede modificarse después de correrlo.

**Configuración congelada:** backbone actual · 8 bloques disjuntos de 16 columnas de la matriz
desplegada · 8 cabezas logísticas promediadas · geometría 32 features · umbral 0,50 · ventanas
k=5 span≤7 · preprocesamiento y cabezas ajustados sobre el pool Train+Validation completo.

**Dos números declarados de antemano:** cobertura **1,00** y cobertura **0,60**. Se reporta la
curva completa, pero **no se destacará ningún otro punto**.

**Predicción registrada antes de mirar:** OOF dio accuracy 0,6849 y BalAcc 0,7003 a cobertura
0,60. Tras seis decisiones sobre OOF, el Test debería venir **por debajo**, con IC de ~±7 pts
sobre 19 participantes. Si diera por encima, es suerte de muestreo y se reporta así.

In [ ]:
import sys, os, json, collections, time
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/SparkleV2'
else:
    PROJECT_DIR = os.environ.get('SPARKLE_PROJECT_DIR', './SparkleV2')

CACHE_32  = f'{PROJECT_DIR}/dataset_final_fase3/dataset_fase3_2_alineado.npz'
CACHE_OLD = f'{PROJECT_DIR}/dataset_final_fase3/dataset_fase3_5_raw1280.npz'
PROJ_ORIG = f'{PROJECT_DIR}/checkpoints_fase3_2/proyeccion_aleatoria_original.npz'
OUT_DIR   = f'{PROJECT_DIR}/fase3_6_cierre'
RESULT    = f'{OUT_DIR}/TEST_OFICIAL_RESULTADO.json'
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------- SALVAGUARDA (regla 1) ----------------
if os.path.exists(RESULT):
    raise SystemExit(
        f'\n⛔ Ya existe {RESULT}.\nEl Test oficial de Fase 3.6 YA fue evaluado. '
        'Reejecutar exige borrar ese archivo a mano, y esa acción queda registrada.\n'
        'Si el objetivo es solo releer los números, abrir el JSON.')
print('Salvaguarda OK: el Test de Fase 3.6 no fue evaluado todavía.')

import numpy as np, pandas as pd
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt

# ------- CONGELADOS por la Enmienda 5 -------
SEED, UMBRAL = 42, 0.50
D_PROJ, N_PROY, K, SPAN_MAX = 16, 8, 5, 7
COBERTURAS = np.round(np.arange(1.00, 0.499, -0.05), 2)
COB_DESTACADAS = [1.00, 0.60]        # declaradas ANTES de mirar
N_BOOT, N_FRAMES, DIM_RAW = 5000, 60, 1280
DELTA_FEATURES = ['ear_avg', 'yaw', 'pitch', 'mar', 'jawOpen', 'eyeBlinkLeft', 'eyeBlinkRight',
                  'eyeLookInLeft', 'eyeLookOutLeft', 'browInnerUp']
OOF_ESPERADO = {'accuracy_c060': 0.6849, 'balacc_c060': 0.7003}

d32 = np.load(CACHE_32, allow_pickle=True); dold = np.load(CACHE_OLD, allow_pickle=True)
Wo = np.load(PROJ_ORIG); W_ORIG, B_ORIG = Wo['W'], Wo['b']
assert W_ORIG.shape == (DIM_RAW, D_PROJ*N_PROY)
FEATS22 = [str(f) for f in d32['feature_names']]
ids_of = lambda a: np.array([str(c) for c in a])
for k in ('tr', 'va', 'te'):
    assert (ids_of(d32[f'id_{k}']) == ids_of(dold[f'id_{k}'])).all()

Xg_pool = np.concatenate([d32['Xg_tr'], d32['Xg_va']])
Xa_pool = np.concatenate([dold['Xr_tr'], dold['Xr_va']])
y_pool  = (np.concatenate([d32['y3_tr'], d32['y3_va']]) > 0).astype(np.int64)
Xg_te, Xa_te = d32['Xg_te'], dold['Xr_te']
y_te_full = (d32['y3_te'] > 0).astype(np.int64); id_te = ids_of(d32['id_te'])
print(f'Pool {len(y_pool)} clips (ajuste) · Test {len(y_te_full)} clips '
      f'· {len(set(c[:6] for c in id_te))} participantes en Test')

## 1. Utilidades — idénticas a todas las fases anteriores

In [ ]:
def interpolate_sequence(seq):
    df = pd.DataFrame(seq).interpolate(method='linear', axis=0, limit_direction='both')
    return df.ffill().bfill().values.astype(np.float32)

def clean(X, medians=None, thresh=0.30):
    keep, out = [], []
    for i in range(len(X)):
        if np.isnan(X[i]).any(axis=1).mean() > thresh:
            continue
        s = interpolate_sequence(X[i])
        if np.isnan(s).any() and medians is not None:
            idx = np.where(np.isnan(s)); s[idx] = np.take(medians, idx[1])
        out.append(s); keep.append(i)
    return np.stack(out), np.array(keep)

delta_idx = [FEATS22.index(f) for f in DELTA_FEATURES]
def add_deltas(X):
    d_ = np.zeros_like(X[:, :, delta_idx]); d_[:, 1:, :] = np.diff(X[:, :, delta_idx], axis=1)
    return np.concatenate([X, d_], axis=-1)

def resumir(X):
    X = np.nan_to_num(X); return np.hstack([X.mean(axis=1), X.std(axis=1)])

def metricas(y, p, umbral=UMBRAL):
    pred = (p > umbral).astype(int); triv = max((y == 0).mean(), (y == 1).mean())
    acc = accuracy_score(y, pred)
    return {'n': int(len(y)), 'trivial': round(float(triv), 4),
            'accuracy': round(float(acc), 4), 'brecha': round(float(acc - triv), 4),
            'f1_macro': round(float(f1_score(y, pred, average='macro')), 4),
            'balanced_acc': round(float(balanced_accuracy_score(y, pred)), 4),
            'auc': round(float(roc_auc_score(y, p)), 4)}

def _ba(y, ok):
    n0, n1 = (y == 0).sum(), (y == 1).sum()
    return np.nan if (n0 == 0 or n1 == 0) else 0.5*(ok[y==0].sum()/n0 + ok[y==1].sum()/n1)

## 2. Ajuste del artefacto final sobre el pool completo

Preprocesamiento y cabezas se ajustan con **Train+Validation**. El Test no interviene.

In [ ]:
g_med = np.nanmedian(Xg_pool.reshape(-1, 22), axis=0)
Xg_p, kgp = clean(Xg_pool, g_med)
Ea_p, kep = clean(Xa_pool.astype(np.float32), None, thresh=0.5)
kp = np.intersect1d(kgp, kep)
ig = {k: i for i, k in enumerate(kgp)}; ie = {k: i for i, k in enumerate(kep)}
gsc = StandardScaler().fit(add_deltas(Xg_p).reshape(-1, 32))
sg = lambda X: gsc.transform(X.reshape(-1, 32)).reshape(len(X), N_FRAMES, 32)
Rg_p = resumir(sg(add_deltas(Xg_p))[[ig[k] for k in kp]])
Ea_k = Ea_p[[ie[k] for k in kp]]; y_fit = y_pool[kp]

cabezas = []
for j in range(N_PROY):
    cols = slice(j*D_PROJ, (j+1)*D_PROJ)
    R = np.hstack([Rg_p, resumir(Ea_k @ W_ORIG[:, cols] + B_ORIG[cols])])
    sc = StandardScaler().fit(R)
    lr = LogisticRegression(max_iter=5000, class_weight='balanced', random_state=SEED)
    lr.fit(sc.transform(R), y_fit)
    cabezas.append((sc, lr))
print(f'{N_PROY} cabezas ajustadas sobre {len(y_fit)} clips del pool · '
      f'{sum(l.coef_.size + 1 for _, l in cabezas)} parámetros de decisión')

## 3. ⚠️ APERTURA DEL TEST OFICIAL — evaluación única

In [ ]:
print('=' * 62); print('ABRIENDO EL TEST OFICIAL — evaluación 1 de 1 para Fase 3.6'); print('=' * 62)

Xg_c, kgc = clean(Xg_te, g_med)
Ea_c, kec = clean(Xa_te.astype(np.float32), None, thresh=0.5)
kc = np.intersect1d(kgc, kec)
jg = {k: i for i, k in enumerate(kgc)}; je = {k: i for i, k in enumerate(kec)}
Rg_c = resumir(sg(add_deltas(Xg_c))[[jg[k] for k in kc]])
Ea_c = Ea_c[[je[k] for k in kc]]
y_c = y_te_full[kc]; ids_c = id_te[kc]; part_c = np.array([c[:6] for c in ids_c])

P = []
for j, (sc, lr) in enumerate(cabezas):
    cols = slice(j*D_PROJ, (j+1)*D_PROJ)
    R = np.hstack([Rg_c, resumir(Ea_c @ W_ORIG[:, cols] + B_ORIG[cols])])
    P.append(lr.predict_proba(sc.transform(R))[:, 1])
p_c = np.mean(P, axis=0)
print(f'Test: {len(y_c)} clips · {len(set(part_c))} participantes · '
      f'trivial por clip {max((y_c==0).mean(), (y_c==1).mean()):.4f}')

## 4. Ventanas y curva riesgo-cobertura

In [ ]:
def parse(cid):
    c = os.path.splitext(str(cid).strip())[0]; return c[:6], int(c[6:])

def ventanas_con_huecos(ids, k=K, span_max=SPAN_MAX):
    por = collections.defaultdict(list)
    for i, cid in enumerate(ids):
        s, t = parse(cid); por[s].append((t, i))
    wins = []
    for s, pares in por.items():
        pares.sort(); i = 0
        while i + k - 1 < len(pares):
            v = pares[i:i+k]
            if v[-1][0] - v[0][0] <= span_max - 1:
                wins.append([j for _, j in v]); i += k
            else:
                i += 1
    return wins

wins = ventanas_con_huecos(ids_c)
for w in wins:
    assert len({parse(ids_c[i])[0] for i in w}) == 1
    ts = sorted(parse(ids_c[i])[1] for i in w); assert ts[-1]-ts[0] <= SPAN_MAX-1
y_w = np.array([int(y_c[w].sum()*2 > len(w)) for w in wins])
p_w = np.array([float(p_c[w].mean()) for w in wins])
part_w = np.array([parse(ids_c[w[0]])[0] for w in wins])
print(f'{len(wins)} ventanas · {len(set(part_w))} participantes · '
      f'cobertura {sum(len(w) for w in wins)/len(ids_c):.1%}')

def curva(p, y):
    conf = np.abs(p - UMBRAL); orden = np.argsort(-conf); filas = []
    for c in COBERTURAS:
        k = max(1, int(round(c*len(p)))); s = orden[:k]
        filas.append({'cobertura': float(c), **metricas(y[s], p[s])})
    return pd.DataFrame(filas)

df_w, df_c = curva(p_w, y_w), curva(p_c, y_c)
print('\n=== TEST OFICIAL · POR VENTANA ==='); print(df_w.to_string(index=False))
print('\n=== TEST OFICIAL · POR CLIP ===');    print(df_c.to_string(index=False))

## 5. IC bootstrap clusterizado por los 19 participantes de Test

In [ ]:
def boot(p, y, part, c, n_boot=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed); parts = np.unique(part)
    idx = {q: np.where(part == q)[0] for q in parts}; conf = np.abs(p - UMBRAL)
    acc, ba, br = (np.empty(n_boot) for _ in range(3))
    for i in range(n_boot):
        j = np.concatenate([idx[q] for q in rng.choice(parts, size=len(parts), replace=True)])
        j = j[np.argsort(-conf[j])[:max(1, int(round(c*len(j))))]]
        ys = y[j]; ok = ((p[j] > UMBRAL).astype(int) == ys)
        acc[i] = ok.mean(); ba[i] = _ba(ys, ok)
        br[i] = acc[i] - max((ys == 0).mean(), (ys == 1).mean())
    pct = lambda v: [round(float(x), 4) for x in np.percentile(v[~np.isnan(v)], [2.5, 97.5])]
    return {'accuracy_IC95': pct(acc), 'balacc_IC95': pct(ba), 'brecha_IC95': pct(br)}

destacados = {}
for c in COB_DESTACADAS:
    fw = df_w[df_w.cobertura == c].iloc[0]; fc = df_c[df_c.cobertura == c].iloc[0]
    rw = boot(p_w, y_w, part_w, c); rc = boot(p_c, y_c, part_c, c)
    destacados[f'c{c:.2f}'] = {'ventana': {**fw.to_dict(), **rw},
                               'clip': {**fc.to_dict(), **rc}}
    print(f'\n--- COBERTURA {c:.2f} (declarada de antemano) ---')
    print(f'VENTANA: acc {fw.accuracy:.4f} {rw["accuracy_IC95"]} · trivial {fw.trivial:.4f} · '
          f'brecha {fw.brecha:+.4f} {rw["brecha_IC95"]}')
    print(f'         BalAcc {fw.balanced_acc:.4f} {rw["balacc_IC95"]} · F1M {fw.f1_macro:.4f} · '
          f'AUC {fw.auc:.4f} · n={int(fw.n)}')
    print(f'CLIP   : acc {fc.accuracy:.4f} {rc["accuracy_IC95"]} · trivial {fc.trivial:.4f} · '
          f'brecha {fc.brecha:+.4f} · BalAcc {fc.balanced_acc:.4f} {rc["balacc_IC95"]}')

## 6. Contraste con la predicción registrada antes de abrir

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(df_w.cobertura, df_w.accuracy, 'o-', label='accuracy Test (ventana)')
plt.plot(df_w.cobertura, df_w.trivial, 's--', alpha=.6, label='trivial')
plt.plot(df_w.cobertura, df_w.balanced_acc, '^-', alpha=.7, label='balanced acc')
plt.axhline(0.70, color='r', ls=':', label='objetivo 0,70')
for c in COB_DESTACADAS:
    plt.axvline(c, color='gray', ls=':')
plt.gca().invert_xaxis(); plt.legend(fontsize=8); plt.xlabel('cobertura')
plt.title('Test oficial — curva riesgo-cobertura')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/test_curva.png', dpi=120); plt.show()

acc_test = destacados['c0.60']['ventana']['accuracy']
ba_test  = destacados['c0.60']['ventana']['balanced_acc']
print(f'A cobertura 0,60 — OOF esperaba accuracy {OOF_ESPERADO["accuracy_c060"]:.4f} / '
      f'BalAcc {OOF_ESPERADO["balacc_c060"]:.4f}')
print(f'                   Test dio          accuracy {acc_test:.4f} / BalAcc {ba_test:.4f}')
print(f'                   diferencia        {acc_test-OOF_ESPERADO["accuracy_c060"]:+.4f} / '
      f'{ba_test-OOF_ESPERADO["balacc_c060"]:+.4f}')
if acc_test > OOF_ESPERADO['accuracy_c060']:
    print('\n⚠️  El Test dio POR ENCIMA del OOF. Con 19 participantes eso es suerte de muestreo, '
          'no una mejora. Se reporta así (Enmienda 5).')
print(f'\n¿Alcanza el objetivo de 0,70 de accuracy? '
      f'{"SÍ" if acc_test >= 0.70 else "NO"} (a cobertura 0,60)')

## 7. Persistencia del resultado y del artefacto

In [ ]:
json.dump({'protocolo': 'Protocolo_Fase36_Camino_al_70.md · Enmienda 5',
           'fecha': pd.Timestamp.now().isoformat(),
           'evaluaciones_test_fase3_6': 1,
           'config': {'backbone': 'MobileNetV2 alpha=0.35 @112 (desplegado)',
                      'proyeccion': '8 bloques disjuntos de 16 col. de la matriz original',
                      'cabezas': N_PROY, 'umbral': UMBRAL, 'k': K, 'span_max': SPAN_MAX},
           'oof_esperado': OOF_ESPERADO,
           'destacados': destacados,
           'curva_ventana': df_w.to_dict('records'),
           'curva_clip': df_c.to_dict('records')},
          open(RESULT, 'w'), indent=2, ensure_ascii=False, default=str)

np.savez(f'{OUT_DIR}/modelo_final_ensemble.npz',
         W=W_ORIG, b=B_ORIG, geom_medians=g_med,
         geom_scaler_mean=gsc.mean_, geom_scaler_scale=gsc.scale_,
         **{f'sc{j}_mean': sc.mean_ for j, (sc, _) in enumerate(cabezas)},
         **{f'sc{j}_scale': sc.scale_ for j, (sc, _) in enumerate(cabezas)},
         **{f'lr{j}_coef': lr.coef_ for j, (_, lr) in enumerate(cabezas)},
         **{f'lr{j}_int': lr.intercept_ for j, (_, lr) in enumerate(cabezas)})
print(f'Resultado en {RESULT}')
print(f'Artefacto en {OUT_DIR}/modelo_final_ensemble.npz')
print('\n**EVALUACIONES SOBRE EL TEST OFICIAL EN FASE 3.6: 1. La compuerta queda cerrada.**')

## 8. Después de esto

1. Registrar en `Fase3_Registro_de_Resultados.md` como §16, con la curva completa y **solo**
   los dos puntos declarados destacados.
2. Corregir `Ficha_Tecnica_Modelos.md` y `Seccion_Tesis_Fase3.md`: el aporte de apariencia de
   +3,0 pts (§11.6), el IC i.i.d. (§9.6, §12.5) y el 0,6374 sin dispersión (§9.7).
3. Implementar la abstención en `detector.js` si se adopta la cobertura 0,60: estado
   `sin_juicio` cuando `|p − 0,5|` cae bajo el umbral de confianza congelado, en vez de mostrar
   un porcentaje que el modelo no sostiene.
4. **No volver a correr este notebook.** La salvaguarda lo impide; borrarla deja constancia.